### Imports

In [1]:
from pathlib import Path

import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text
import urllib.parse

print("Pandas:", pd.__version__)
print("PyODBC:", pyodbc.version)

Pandas: 3.0.5
PyODBC: 5.3.0


### Check installed ODBC drivers

In [2]:
drivers = pyodbc.drivers()

print("Installed ODBC Drivers:")

for driver in drivers:
    print("-", driver)

Installed ODBC Drivers:
- SQL Server
- SQL Server Native Client RDA 11.0
- ODBC Driver 17 for SQL Server
- SnowflakeDSIIDriver
- Microsoft Access Driver (*.mdb, *.accdb)
- Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)
- Microsoft Access Text Driver (*.txt, *.csv)
- Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)


### Define SQL Server details

In [6]:
SERVER = r"localhost\SQLEXPRESS"
DATABASE = "CustomerSalesInventoryOps"

print("Server:", SERVER)
print("Database:", DATABASE)

Server: localhost\SQLEXPRESS
Database: CustomerSalesInventoryOps


### Choose the SQL driver automatically

In [7]:
available_drivers = pyodbc.drivers()

if "ODBC Driver 18 for SQL Server" in available_drivers:
    DRIVER = "ODBC Driver 18 for SQL Server"

elif "ODBC Driver 17 for SQL Server" in available_drivers:
    DRIVER = "ODBC Driver 17 for SQL Server"

else:
    raise RuntimeError(
        "No supported SQL Server ODBC driver found."
    )

print("Using driver:", DRIVER)

Using driver: ODBC Driver 17 for SQL Server


### Test direct PyODBC connection

In [8]:
connection_string = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

conn = pyodbc.connect(
    connection_string,
    timeout=10
)

print("PyODBC connection successful.")

PyODBC connection successful.


In [9]:
cursor = conn.cursor()

cursor.execute("""
SELECT
    DB_NAME() AS CurrentDatabase,
    @@SERVERNAME AS ServerName
""")

row = cursor.fetchone()

print("Current Database:", row.CurrentDatabase)
print("SQL Server:", row.ServerName)

cursor.close()
conn.close()

Current Database: CustomerSalesInventoryOps
SQL Server: LAPTOP-5P4TH31J\SQLEXPRESS


### Test SQLAlchemy

In [10]:
params = urllib.parse.quote_plus(
    connection_string
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

print("SQLAlchemy engine created.")

SQLAlchemy engine created.


In [11]:
with engine.connect() as connection:

    result = connection.execute(
        text("""
        SELECT
            DB_NAME() AS CurrentDatabase,
            COUNT(*) AS MetadataRows
        FROM core.ProjectMetadata
        """)
    )

    row = result.fetchone()

    print("Current Database:", row.CurrentDatabase)
    print("ProjectMetadata rows:", row.MetadataRows)

Current Database: CustomerSalesInventoryOps
ProjectMetadata rows: 1


### Load the Parquet analytical master

In [12]:
current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

ANALYTICS_FILE = (
    INTERIM_DIR /
    "analytics_master.parquet"
)

analytics_master = pd.read_parquet(
    ANALYTICS_FILE
)

print(
    "Loaded analytical rows:",
    f"{len(analytics_master):,}"
)

print(
    "Columns:",
    analytics_master.shape[1]
)

Loaded analytical rows: 1,033,036
Columns: 27


### Prepare SQL staging columns

In [13]:
sql_df = analytics_master[
    [
        "invoice",
        "stock_code",
        "description",
        "customer_id",
        "country",
        "source_sheet",
        "quantity",
        "unit_price",
        "transaction_value",
        "invoice_date",
        "invoice_year",
        "invoice_month",
        "year_month",
        "invoice_week",
        "day_of_week",
        "transaction_type",
        "is_cancellation",
        "is_negative_quantity",
        "is_zero_price",
        "is_negative_price",
        "is_missing_customer",
        "is_missing_description",
        "appears_across_sheets",
        "transaction_hash"
    ]
].copy()

In [14]:
sql_df = sql_df.rename(
    columns={
        "invoice": "Invoice",
        "stock_code": "StockCode",
        "description": "Description",
        "customer_id": "CustomerID",
        "country": "Country",
        "source_sheet": "SourceSheet",
        "quantity": "Quantity",
        "unit_price": "UnitPrice",
        "transaction_value": "TransactionValue",
        "invoice_date": "InvoiceDate",
        "invoice_year": "InvoiceYear",
        "invoice_month": "InvoiceMonth",
        "year_month": "YearMonth",
        "invoice_week": "InvoiceWeek",
        "day_of_week": "DayOfWeek",
        "transaction_type": "TransactionType",
        "is_cancellation": "IsCancellation",
        "is_negative_quantity": "IsNegativeQuantity",
        "is_zero_price": "IsZeroPrice",
        "is_negative_price": "IsNegativePrice",
        "is_missing_customer": "IsMissingCustomer",
        "is_missing_description": "IsMissingDescription",
        "appears_across_sheets": "AppearsAcrossSheets",
        "transaction_hash": "TransactionHash"
    }
)

### Fix the transaction hash for SQL Server

In [15]:
sql_df["TransactionHash"] = (
    sql_df["TransactionHash"]
    .astype("string")
)

print(sql_df.dtypes)

display(
    sql_df.head()
)

Invoice                         string
StockCode                       string
Description                     string
CustomerID                       Int64
Country                         string
SourceSheet                     string
Quantity                         int64
UnitPrice                      float64
TransactionValue               float64
InvoiceDate             datetime64[us]
InvoiceYear                      int32
InvoiceMonth                     int32
YearMonth                       string
InvoiceWeek                     string
DayOfWeek                          str
TransactionType                    str
IsCancellation                 boolean
IsNegativeQuantity                bool
IsZeroPrice                       bool
IsNegativePrice                   bool
IsMissingCustomer                 bool
IsMissingDescription              bool
AppearsAcrossSheets               bool
TransactionHash                 string
dtype: object


,Invoice,StockCode,Description,CustomerID,Country,SourceSheet,Quantity,UnitPrice,TransactionValue,InvoiceDate,...,DayOfWeek,TransactionType,IsCancellation,IsNegativeQuantity,IsZeroPrice,IsNegativePrice,IsMissingCustomer,IsMissingDescription,AppearsAcrossSheets,TransactionHash
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,13085,United Kingdom,Year 2009-2010,12,6.95,83.4,2009-12-01 07:45:00,...,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,7243821383875921504
1,489434,79323P,PINK CHERRY LIGHTS,13085,United Kingdom,Year 2009-2010,12,6.75,81.0,2009-12-01 07:45:00,...,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,17514895690039914094
2,489434,79323W,WHITE CHERRY LIGHTS,13085,United Kingdom,Year 2009-2010,12,6.75,81.0,2009-12-01 07:45:00,...,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,12457407259001370697
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",13085,United Kingdom,Year 2009-2010,48,2.10,100.8,2009-12-01 07:45:00,...,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,18012193762395552222
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,13085,United Kingdom,Year 2009-2010,24,1.25,30.0,2009-12-01 07:45:00,...,Tuesday,Merchandise Sale,False,False,False,False,False,False,False,8093434425875725190


### Create a 1,000-row test sample

In [16]:
test_df = sql_df.head(1000).copy()

print(
    "Test rows:",
    len(test_df)
)

Test rows: 1000


### Make sure staging table is empty

In [17]:
with engine.begin() as connection:
    connection.execute(
        text("""
        TRUNCATE TABLE staging.Transactions;
        """)
    )

print("Staging table cleared.")

Staging table cleared.


### Load only 1,000 rows

In [18]:
test_df.to_sql(
    name="Transactions",
    con=engine,
    schema="staging",
    if_exists="append",
    index=False,
    chunksize=1000
)

print("1,000-row test load completed.")

1,000-row test load completed.


### Verify the test load from Python

In [21]:
test_validation = pd.read_sql(
    """
    SELECT
        COUNT(*) AS TransactionCount,
        COUNT(DISTINCT Invoice) AS UniqueInvoices,
        MIN(InvoiceDate) AS MinInvoiceDate,
        MAX(InvoiceDate) AS MaxInvoiceDate
    FROM staging.Transactions;
    """,
    engine
)

display(test_validation)

,TransactionCount,UniqueInvoices,MinInvoiceDate,MaxInvoiceDate
0,1000,62,2009-12-01 07:45:00,2009-12-01 12:30:00


### Inspect actual SQL rows

In [22]:
sample_sql_rows = pd.read_sql(
    """
    SELECT TOP 10
        StagingTransactionID,
        Invoice,
        StockCode,
        Description,
        Quantity,
        UnitPrice,
        CustomerID,
        Country,
        TransactionType
    FROM staging.Transactions
    ORDER BY StagingTransactionID
    """,
    engine
)

display(sample_sql_rows)

,StagingTransactionID,Invoice,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,TransactionType
0,1,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,13085,United Kingdom,Merchandise Sale
1,2,489434,79323P,PINK CHERRY LIGHTS,12,6.75,13085,United Kingdom,Merchandise Sale
2,3,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,13085,United Kingdom,Merchandise Sale
3,4,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,13085,United Kingdom,Merchandise Sale
4,5,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,13085,United Kingdom,Merchandise Sale
5,6,489434,22064,PINK DOUGHNUT TRINKET POT,24,1.65,13085,United Kingdom,Merchandise Sale
6,7,489434,21871,SAVE THE PLANET MUG,24,1.25,13085,United Kingdom,Merchandise Sale
7,8,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,13085,United Kingdom,Merchandise Sale
8,9,489435,22350,CAT BOWL,12,2.55,13085,United Kingdom,Merchandise Sale
9,10,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,3.75,13085,United Kingdom,Merchandise Sale


# Full SQL staging load

### Clear the test rows

In [23]:
with engine.begin() as connection:
    connection.execute(
        text("""
        TRUNCATE TABLE staging.Transactions;
        """)
    )

print("Staging table cleared.")

Staging table cleared.


In [24]:
empty_check = pd.read_sql(
    """
    SELECT COUNT(*) AS TransactionCount
    FROM staging.Transactions;
    """,
    engine
)

display(empty_check)

,TransactionCount
0,0


### Prepare the full load

In [25]:
print(
    "Rows ready for SQL load:",
    f"{len(sql_df):,}"
)

Rows ready for SQL load: 1,033,036


### Full batch load with timing

In [26]:
import time

start_time = time.time()

batch_size = 25000
total_rows = len(sql_df)

for start in range(0, total_rows, batch_size):

    end = min(
        start + batch_size,
        total_rows
    )

    batch = sql_df.iloc[start:end]

    batch.to_sql(
        name="Transactions",
        con=engine,
        schema="staging",
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print(
        f"Loaded {end:,} / {total_rows:,} rows"
    )

elapsed_seconds = time.time() - start_time

print()
print("Full staging load complete.")
print(
    f"Elapsed time: "
    f"{elapsed_seconds / 60:.2f} minutes"
)

Loaded 25,000 / 1,033,036 rows
Loaded 50,000 / 1,033,036 rows
Loaded 75,000 / 1,033,036 rows
Loaded 100,000 / 1,033,036 rows
Loaded 125,000 / 1,033,036 rows
Loaded 150,000 / 1,033,036 rows
Loaded 175,000 / 1,033,036 rows
Loaded 200,000 / 1,033,036 rows
Loaded 225,000 / 1,033,036 rows
Loaded 250,000 / 1,033,036 rows
Loaded 275,000 / 1,033,036 rows
Loaded 300,000 / 1,033,036 rows
Loaded 325,000 / 1,033,036 rows
Loaded 350,000 / 1,033,036 rows
Loaded 375,000 / 1,033,036 rows
Loaded 400,000 / 1,033,036 rows
Loaded 425,000 / 1,033,036 rows
Loaded 450,000 / 1,033,036 rows
Loaded 475,000 / 1,033,036 rows
Loaded 500,000 / 1,033,036 rows
Loaded 525,000 / 1,033,036 rows
Loaded 550,000 / 1,033,036 rows
Loaded 575,000 / 1,033,036 rows
Loaded 600,000 / 1,033,036 rows
Loaded 625,000 / 1,033,036 rows
Loaded 650,000 / 1,033,036 rows
Loaded 675,000 / 1,033,036 rows
Loaded 700,000 / 1,033,036 rows
Loaded 725,000 / 1,033,036 rows
Loaded 750,000 / 1,033,036 rows
Loaded 775,000 / 1,033,036 rows
Loaded 800,

### Final SQL row reconciliation

In [28]:
sql_count = pd.read_sql(
    """
    SELECT COUNT(*) AS TransactionCount
    FROM staging.Transactions;
    """,
    engine
)

display(sql_count)

,TransactionCount
0,1033036


In [29]:
python_rows = len(sql_df)

sql_rows = int(
    sql_count.loc[
        0,
        "TransactionCount"
    ]
)

print(
    "Python analytical rows:",
    f"{python_rows:,}"
)

print(
    "SQL staging rows:",
    f"{sql_rows:,}"
)

print(
    "Reconciliation:",
    python_rows == sql_rows
)

Python analytical rows: 1,033,036
SQL staging rows: 1,033,036
Reconciliation: True


### Validate business totals

In [30]:
sql_validation = pd.read_sql(
    """
    SELECT
        COUNT(*) AS TransactionCount,
        COUNT(DISTINCT Invoice) AS UniqueInvoices,
        COUNT(DISTINCT CustomerID) AS UniqueCustomers,
        COUNT(DISTINCT StockCode) AS UniqueStockCodes,

        MIN(InvoiceDate) AS MinInvoiceDate,
        MAX(InvoiceDate) AS MaxInvoiceDate,

        SUM(
            CASE
                WHEN TransactionType = 'Merchandise Sale'
                THEN 1
                ELSE 0
            END
        ) AS MerchandiseSaleRows,

        SUM(
            CASE
                WHEN TransactionType = 'Merchandise Cancellation'
                THEN 1
                ELSE 0
            END
        ) AS CancellationRows

    FROM staging.Transactions;
    """,
    engine
)

display(sql_validation)

,TransactionCount,UniqueInvoices,UniqueCustomers,UniqueStockCodes,MinInvoiceDate,MaxInvoiceDate,MerchandiseSaleRows,CancellationRows
0,1033036,53628,5942,5131,2009-12-01 07:45:00,2011-12-09 12:50:00,1003439,17932


### Validate merchandise revenue in SQL

In [31]:
sql_revenue = pd.read_sql(
    """
    SELECT
        SUM(
            CASE
                WHEN TransactionType = 'Merchandise Sale'
                THEN TransactionValue
                ELSE 0
            END
        ) AS MerchandiseRevenue
    FROM staging.Transactions;
    """,
    engine
)

display(sql_revenue)

,MerchandiseRevenue
0,1.964563e+07


### Validate transaction-type counts

In [32]:
sql_transaction_types = pd.read_sql(
    """
    SELECT
        TransactionType,
        COUNT(*) AS TransactionCount
    FROM staging.Transactions
    GROUP BY TransactionType
    ORDER BY TransactionCount DESC;
    """,
    engine
)

display(sql_transaction_types)

,TransactionType,TransactionCount
0,Merchandise Sale,1003439
1,Merchandise Cancellation,17932
2,Shipping / Service Charge,3788
3,Inventory / Operational Adjustment,3393
4,Zero-Price Product Movement,2594
5,Manual / System Adjustment,1457
6,Discount,173
7,Financial Adjustment,142
8,Sample,101
9,Test Transaction,17
